In [1]:
#imports
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from datasets import load_dataset
from torch.nn.utils.rnn import pad_sequence
import re
import string
from collections import Counter
import gc
import numpy as np
import torch.nn.functional as F

/Users/rubenhayrapetyan/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [2]:
#data shapes
imdb = load_dataset("imdb")
sst2 = {
    "train": load_dataset("SetFit/sst2", split="train"),
    "test": load_dataset("SetFit/sst2", split="test")
}
# === Preprocessing ===
def clean_text(text):
    text = text.lower()
    text = re.sub(f"[{re.escape(string.punctuation)}]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

# === Tokenization & Vocab ===
def tokenize(text):
    return clean_text(text).split()

# Build vocab from IMDB
counter = Counter()
for example in imdb["train"]:
    counter.update(tokenize(example["text"]))

# Only keep words with min_freq ≥ 5
min_freq = 5
vocab = {"<pad>": 0, "<unk>": 1}
for word, freq in counter.items():
    if freq >= min_freq:
        vocab[word] = len(vocab)
inv_vocab = {v: k for k, v in vocab.items()}
PAD_IDX = vocab["<pad>"]
UNK_IDX = vocab["<unk>"]

def text_pipeline(text, max_len=256):
    tokens = tokenize(text)
    ids = [vocab.get(token, UNK_IDX) for token in tokens[:max_len]]
    return torch.tensor(ids, dtype=torch.long)

# === Dataset Wrappers ===
class TextDataset(Dataset):
    def __init__(self, hf_dataset, label_transform, max_len=256):
        self.data = hf_dataset
        self.label_transform = label_transform
        self.max_len = max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        text_field = item["text"] if "text" in item else item["content"]
        tokens = text_pipeline(text_field, max_len=self.max_len)
        label = torch.tensor(self.label_transform(item["label"]), dtype=torch.long)
        return tokens, label

def collate_batch(batch):
    texts, labels = zip(*batch)
    padded = pad_sequence(texts, batch_first=True, padding_value=PAD_IDX)
    return padded, torch.stack(labels)

imdb_dataset = TextDataset(imdb["train"], lambda x: x)
sst2_dataset = TextDataset(sst2["train"], lambda x: x)

imdb_loader = DataLoader(imdb_dataset, batch_size=64, shuffle=True, collate_fn=collate_batch)
sst2_loader = DataLoader(sst2_dataset, batch_size=256, shuffle=True, collate_fn=collate_batch)

Repo card metadata block was not found. Setting CardData to empty.
Repo card metadata block was not found. Setting CardData to empty.


In [3]:
class TextClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim=100, hidden_dim=128, num_classes=2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_IDX)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        x = self.embedding(x)
        _, (h_n, _) = self.lstm(x)
        return self.fc(h_n[-1])

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = TextClassifier(len(vocab), num_classes=2).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)


In [4]:
#sliced wasserstein
from sliced_wasserstein import sliced_wasserstein_distance

def feature_extractor(loader, max_samples=1000):
    features = []
    model.eval()
    sample_count = 0
    
    with torch.no_grad():
        for x, _ in loader:
            if sample_count >= max_samples:
                break
            
            feature = model(x)
            features.append(feature.numpy())
            sample_count += x.shape[0]
        return np.concatenate(features, axis=0)
    
imdb_features = feature_extractor(imdb_loader)
print("Extracted imdb features")
 
sst2_features = feature_extractor(sst2_loader)
print("Extracted amazon features")

print("Sliced Wasserstein Distance between amazon and imdb:", 
      sliced_wasserstein_distance(sst2_features, imdb_features))

Extracted imdb features
Extracted amazon features
Sliced Wasserstein Distance between amazon and imdb: 0.018859955144466714


In [5]:
#JS divergence
from sklearn.feature_extraction.text import CountVectorizer

def KL_numpy(a, b):
    a = np.asarray(a, dtype=np.float64)
    b = np.asarray(b, dtype=np.float64)
    
    epsilon = 1e-10
    b = np.where(b == 0, epsilon, b)
    
    return np.sum(np.where(a != 0, a * np.log(a / b), 0))

imdb = load_dataset("imdb", split='train[:10000]')
sst2 = load_dataset("SetFit/sst2", split='train[:10000]')

texts_imdb = [x['text'] for x in imdb]
texts_sst2 = [x['text'] for x in sst2]

vectorizer = CountVectorizer(max_features=5000, stop_words='english')
vectorizer.fit(texts_imdb + texts_sst2)

X_imdb = vectorizer.transform(texts_imdb).toarray().astype(np.float32)
X_sst2 = vectorizer.transform(texts_sst2).toarray().astype(np.float32)

def to_prob_dist(data):
    data = data + 1e-10
    return data / data.sum(axis=1, keepdims=True)

datasets = {
    'IMDB': X_imdb,
    'SST2': X_sst2
}

prob_datasets = {}
avg_datasets = {}

for name, data in datasets.items():
    prob_datasets[name] = to_prob_dist(data)
    avg_datasets[name] = prob_datasets[name].mean(axis=0)

print("Jensen-Shannon Divergence between dataset pairs:")
for name1 in datasets.keys():
    for name2 in datasets.keys():
        if name1 != name2:
            m = (avg_datasets[name1] + avg_datasets[name2]) * 0.5
            js = 0.5 * KL_numpy(avg_datasets[name1], m) + 0.5 * KL_numpy(avg_datasets[name2], m)
            print(f"{name1} vs {name2}: {js:.12f}")

Repo card metadata block was not found. Setting CardData to empty.


Jensen-Shannon Divergence between dataset pairs:
IMDB vs SST2: 0.114919251006
SST2 vs IMDB: 0.114919251006


In [6]:
#cosine similarity + l2 norm
import torch.nn.functional as F

def get_gradient_vector(model, dataloader):
    model.train()
    model.zero_grad()
    x, y = next(iter(dataloader))
    x, y = x.to(device), y.to(device)

    out = model(x)
    loss = F.cross_entropy(out, y)
    loss.backward()

    grads = []
    for param in model.parameters():
        if param.grad is not None:
            grads.append(param.grad.view(-1))
    return torch.cat(grads)

grad_imdb = get_gradient_vector(model, imdb_loader)
grad_sst2 = get_gradient_vector(model, sst2_loader)

# === L2 + Cosine ===
l2_diff = torch.norm(grad_imdb - grad_sst2).item()
cos_sim = F.cosine_similarity(grad_imdb.unsqueeze(0), grad_sst2.unsqueeze(0)).item()

print(f"L2 norm of gradient difference: {l2_diff:.12f}")
print(f"Cosine similarity between gradients: {cos_sim:.12f}")

L2 norm of gradient difference: 0.187497809529
Cosine similarity between gradients: -0.577404320240


In [7]:
#mmd
from discrepancies import MaximumMeanDiscrepancy
from kernels import LaplacianKernel

imdb = load_dataset("imdb", split='train[:1000]')
sst2 = load_dataset("SetFit/sst2", split='train[:1000]')

texts_imdb = [x['text'] for x in imdb]
texts_sst2 = [x['text'] for x in sst2]

vectorizer = CountVectorizer(max_features=5000, stop_words='english')
vectorizer.fit(texts_imdb + texts_sst2)

X_imdb = vectorizer.transform(texts_imdb).toarray().astype(np.float32)
X_sst2 = vectorizer.transform(texts_sst2).toarray().astype(np.float32)

mmd = MaximumMeanDiscrepancy(kernel=LaplacianKernel(1e-2))
print("MMD between imdb and amazon:", mmd.compute(X_imdb, X_sst2))

Repo card metadata block was not found. Setting CardData to empty.


MMD between imdb and amazon: 0.25180635


In [8]:
#hellinger

def hellinger_np(p, q):
    return np.sqrt(0.5 * np.sum((np.sqrt(p) - np.sqrt(q)) ** 2))

imdb = load_dataset("imdb", split='train[:1000]')
sst2 = load_dataset("SetFit/sst2", split='train[:1000]')

texts_imdb = [x['text'] for x in imdb]
texts_sst2 = [x['text'] for x in sst2]

vectorizer = CountVectorizer(max_features=5000, stop_words='english')
vectorizer.fit(texts_imdb + texts_sst2)

X_imdb = vectorizer.transform(texts_imdb).toarray().astype(np.float32)
X_sst2 = vectorizer.transform(texts_sst2).toarray().astype(np.float32)

print("Hellinger Distance between imdb and amazon:", hellinger_np(X_imdb, X_sst2))


Repo card metadata block was not found. Setting CardData to empty.


Hellinger Distance between imdb and amazon: 224.87976
